In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer  # handles missing values

BASE_DIR = "/kaggle/input/siim-isic-melanoma-classification"
TRAIN_PATH = os.path.join(BASE_DIR, "train.csv")
TEST_PATH = os.path.join(BASE_DIR, "test.csv")
SAMPLE_SUB_PATH = os.path.join(BASE_DIR, "sample_submission.csv")

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sub = pd.read_csv(SAMPLE_SUB_PATH)  # correct ordering

# ---------- feature engineering ----------
train_df["sex_site"] = (
    train_df["sex"].fillna("missing")
    + "_"
    + train_df["anatom_site_general_challenge"].fillna("missing")
)
test_df["sex_site"] = (
    test_df["sex"].fillna("missing")
    + "_"
    + test_df["anatom_site_general_challenge"].fillna("missing")
)

train_df["age_sex"] = train_df["age_approx"] * (
    train_df["sex"].fillna("missing") == "male"
).astype(int)
test_df["age_sex"] = test_df["age_approx"] * (
    test_df["sex"].fillna("missing") == "male"
).astype(int)

train_df["age_log"] = np.log1p(train_df["age_approx"])
test_df["age_log"] = np.log1p(test_df["age_approx"])

train_df["sex_missing"] = train_df["sex"].isna().astype(int)
test_df["sex_missing"] = test_df["sex"].isna().astype(int)

train_df["age_sq"] = train_df["age_approx"] ** 2
test_df["age_sq"] = test_df["age_approx"] ** 2

age_bins = [0, 30, 45, 60, 80, np.inf]
train_df["age_bin"] = pd.cut(train_df["age_approx"], bins=age_bins, labels=False)
test_df["age_bin"] = pd.cut(test_df["age_approx"], bins=age_bins, labels=False)

# ensure test has the same columns as train for the pipeline
missing_cols = {"diagnosis", "benign_malignant"} - set(test_df.columns)
for col in missing_cols:
    test_df[col] = np.nan

target_col = "target"
drop_cols = [
    "image_name"
]  # keep patient_id as a feature for now; we'll drop it later in the transformer

X = train_df.drop(columns=[target_col] + drop_cols)
y = train_df[target_col]

# ---------- preprocessing ----------
numeric_features = [
    "age_approx",
    "age_sex",
    "age_log",
    "sex_missing",
    "age_sq",
    # "age_bin" removed – treat as categorical
]

# Exclude high‑cardinality patient_id from the model
categorical_features = [
    c for c in X.columns if c not in numeric_features + ["patient_id"]
]

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                [
                    (
                        "imputer",
                        SimpleImputer(strategy="constant", fill_value="missing"),
                    ),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

# ---------- model ----------
model = LogisticRegression(
    max_iter=1000,
    n_jobs=5,
    class_weight="balanced",
    C=20.0,  # slightly less regularisation to capture more signal
    solver="lbfgs",
)

clf = Pipeline(steps=[("prep", preprocess), ("clf", model)])

# ---------- validation ----------
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf.fit(X_train, y_train)
val_pred = clf.predict_proba(X_val)[:, 1]
print("Validation AUC:", roc_auc_score(y_val, val_pred))

# ---------- final training & prediction ----------
clf.fit(X, y)

X_test = test_df.drop(
    columns=drop_cols
)  # patient_id will be ignored by the transformer
test_pred = clf.predict_proba(X_test)[:, 1]

sub["target"] = test_pred
sub = sub[["image_name", "target"]]



/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


TypeError: Encoders require their input to be uniformly strings or numbers. Got ['float', 'str']

In [2]:
output_path = "submission.csv"
sub.to_csv(output_path, index=False)
print(f"Submission file written to {output_path}")

Submission file written to submission.csv
